In [1]:
%load_ext autoreload
%autoreload 2

from blockymetamaterials.plotting import generate_animation, generate_frames, plot_geometry, generate_patch_collection, generate_mode_images
from blockymetamaterials.utils import SolutionData, EigenmodeData, ControlParams, GeometricalParams, LigamentParams, MechanicalParams, save_data, load_data
from blockymetamaterials.geometry import RotatedSquareGeometry, compute_inertia, DOFsInfo
from blockymetamaterials.energy import build_strain_energy
from blockymetamaterials.analysis_utils import *

from pathlib import Path
from tqdm import tqdm

# import nlopt
import numpy as np

from jax._src.config import config
config.update("jax_enable_x64", True)  # enable float64 type


No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


# Local Functions & Definitions

In [2]:
def get_stiffness_damping_matrices(geometry, control_params, damp_coeffs=np.zeros(3), damptype='hinge', constrained_block_DOF_pairs=np.array([]), state0 = None):
    """
    Returns the stiffness and damping matrices for the system.
    """
    # Initial conditions
    if state0 is None:
        state0 = np.zeros((2, geometry.n_blocks, 3))

    # Geometry
    block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()  # centroid_node_vectors is a function of the initial angle

    # Setup energy function
    potential_energy = build_strain_energy(bond_connectivity=bond_connectivity())

    # Force coefficients
    _stiffness, _damping = linear_response_analysis(
        frequencies=np.array([]),
        displacement=state0[0],
        geometry=geometry,
        energy_fn=potential_energy,
        control_params=control_params,
        constrained_block_DOF_pairs=constrained_block_DOF_pairs,
        drive_amplitude=None,
        drive_angle=None,
        driven_block_DOF_pairs=np.array([]),
        return_force_coeffs=True,
        damp_coeffs=damp_coeffs,
        damptype=damptype
    )

    return _stiffness, _damping

In [3]:
exp_config = experimental_parameters()

# Setup geometry
geometry = RotatedSquareGeometry(
    n1_cells=exp_config['n1_blocks']//2, 
    n2_cells=exp_config['n2_blocks']//2, 
    spacing=exp_config['spacing'], 
    bond_length=exp_config['hinge_length']
)


block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()

print(f"Configuration: {exp_config['n1_blocks']}×{exp_config['n2_blocks']} lattice")
print(f"Total blocks: {exp_config['N']}")

Configuration: 24×16 lattice
Total blocks: 384


# Non-linear Simulation

In [ ]:
exp_freqs = np.arange(2, 36, 1)

angle30 = 30.0 * np.pi/180.0

fieldpath = f'../../data/nonlinear_data/internal_damping/k_stretch_20.59_k_shear_0.83_k_rot_1.68/exp_harmonic_input'
forcepath = f'../../data/nonlinear_data/reaction_forces/exp_harmonic_input/30_deg_sample'

def load_nonlin_funcn(freq, angle):
    tpts = np.load(f'{fieldpath}/{freq}Hz/timepoints.npz')['arr_0']
    fields = np.load(f'{fieldpath}/{freq}Hz/{angle*180/np.pi:.0f}fields.npz')['arr_0']
    forces = np.load(f'{forcepath}/{freq}Hz/reaction_forces.npz')['arr_0']

    return tpts, fields, forces

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle30,
    k_stretch=exp_config['k_stretch'],
    k_rot=exp_config['k_rot'],
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])

Map expressions for conservation analysis :  [z, zb, I*zb, z**2, I*z**2, z*zb, I*z*zb, zb**2, I*zb**2, I*z, 1, I]


In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/nonlinear_30_deg_sample')

for i, freq in enumerate(exp_freqs):
    
    tpts, fields, forces = load_nonlin_funcn(freq, angle30)

    result = analyse_dataset_conservation(
        tpts,
        fields=fields,
        forces=forces,
        tools=tools,
    )
    
    # Save results (ensure directories exist once per save target)
    mom_dir = save_path / 'map_momenta'
    momrate_dir = save_path / 'map_momentum_rates'
    forces_dir = save_path / 'map_ext_forces'
    for d in (mom_dir, momrate_dir, forces_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
    np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
    np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
        
    print(f"Finished processing frequency: {freq} Hz")


np.save(f'{save_path}/map_norms.npy',  tools['map_norm'])

In [ ]:
exp_freqs = np.arange(2, 36, 1)

angle5 = 5.0 * np.pi/180.0


fieldpath = f'../../data/nonlinear_data/internal_damping/k_stretch_20.59_k_shear_0.83_k_rot_1.68/exp_harmonic_input'
forcepath = f'../../data/nonlinear_data/reaction_forces/exp_harmonic_input/5_deg_sample'

def load_nonlin_funcn(freq, angle):
    tpts = np.load(f'{fieldpath}/{freq}Hz/timepoints.npz')['arr_0']
    fields = np.load(f'{fieldpath}/{freq}Hz/{angle*180/np.pi:.0f}fields.npz')['arr_0']
    forces = np.load(f'{forcepath}/{freq}Hz/reaction_forces.npz')['arr_0']

    return tpts, fields, forces

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle5,
    k_stretch=exp_config['k_stretch'],
    k_rot=exp_config['k_rot'],
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/nonlinear_5_deg_sample')

for i, freq in enumerate(exp_freqs):
    
    tpts, fields, forces = load_nonlin_funcn(freq, angle5)

    result = analyse_dataset_conservation(
        tpts,
        fields=fields,
        forces=forces,
        tools=tools,
    )
    
    # Save results (ensure directories exist once per save target)
    mom_dir = save_path / 'map_momenta'
    momrate_dir = save_path / 'map_momentum_rates'
    forces_dir = save_path / 'map_ext_forces'
    for d in (mom_dir, momrate_dir, forces_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
    np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
    np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
        
    print(f"Finished processing frequency: {freq} Hz")

np.save(f'{save_path}/map_norms.npy',  tools['map_norm'])

# Non-linear Simulation Varying $k_{\theta}$ Amplitude $4$ mm

In [ ]:
exp_freqs = np.arange(2, 36, 1)
i_range = np.arange(-8, 6)

amp = '4'

angle30 = 30.0 * np.pi/180.0

fieldpath = f'../../data/nonlinear_data/internal_damping/k_stretch_20.59_k_shear_0.83_sweeping_k_rot'
forcepath = f'../../data/nonlinear_data/reaction_forces/sim_data_harmonic_input_sweeping_krot/30_deg_sample'


def load_nonlin_varykrot_funcn(freq, angle):
    pow_ = 0
    tpts = np.load(f'{fieldpath}/{angle*180/np.pi:.0f}_deg_{freq}.0Hz_{amp}mm_10periods_sine_krotx2power{pow_}/timepoints.npz')['arr_0']

    fields = np.zeros((len(i_range), len(tpts), 2, geometry.n_blocks, 3))  # Initialize array to hold fields for all k_rot
    forces = np.zeros((len(i_range), len(tpts), geometry.n_blocks, 3))     # Initialize array to hold forces for all k_rot
    
    for idx, pow in enumerate(i_range):
        fields[idx] = np.load(f'{fieldpath}/{angle*180/np.pi:.0f}_deg_{freq}.0Hz_{amp}mm_10periods_sine_krotx2power{pow}/{angle30*180/np.pi:.0f}_deg_fields.npz')['arr_0']
        forces[idx] = np.load(f'{forcepath}/{freq}Hz/krotx2power{pow}_amplitude_{amp}_10periods_sine_reaction_forces.npz')['arr_0']

    return tpts, fields, forces


In [ ]:
# Setup analysis tools for varying parameter

k_rot_arr = exp_config['k_rot'] *  2.0**i_range

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle30,
    k_stretch=exp_config['k_stretch'],
    k_rot=k_rot_arr,
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/nonlinear_varying_krot')

d = save_path / 'map_norms'
d.mkdir(parents=True, exist_ok=True)

for idx, pow in enumerate(i_range):
    np.save(f'{save_path}/map_norms/krotx2power{pow}.npy',  tools['map_norm'][idx])

for i, freq in enumerate(exp_freqs):
    print(f"Processing frequency: {freq} Hz ({i+1}/{len(exp_freqs)})")
    
    tpts, fields, forces = load_nonlin_varykrot_funcn(freq)
    
    # Process each k_rot dataset
    for idx, pow in enumerate(i_range):
        print(f"  Analyzing k_rot x 2^{pow} ({idx+1}/{len(i_range)})")
        
        result = analyse_dataset_conservation(
            tpts,
            fields=fields[idx],
            forces=forces[idx],
            tools=slice_tools_for_param(tools, idx),
        )
        # print(f"map_ext_forces shape: {result['map_ext_forces'].shape}")
        # print(f"map_ext_forces dtype: {result['map_ext_forces'].dtype}")
        # print(f"map_ext_forces sample values: {result['map_ext_forces'][:5, :5]}")
        # print(f"Any NaN? {np.isnan(result['map_ext_forces']).any()}")
        
        # Save results (ensure directories exist once per save target)
        mom_dir = save_path / 'map_momenta' / f'krotx2power{pow}_amplitude_{amp}mm'
        momrate_dir = save_path / 'map_momentum_rates' / f'krotx2power{pow}_amplitude_{amp}mm'
        forces_dir = save_path / 'map_ext_forces' / f'krotx2power{pow}_amplitude_{amp}mm'
        for d in (mom_dir, momrate_dir, forces_dir):
            d.mkdir(parents=True, exist_ok=True)
        
        np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
        np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
        np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
        
    print(f"Finished processing frequency: {freq} Hz")

# Non-linear Simulation Varying $k_{\theta}$ Amplitude $10^{-8}$ mm

In [ ]:
lowamp_freqs = np.arange(5, 36, 5)
i_range = np.arange(-8, 6)

amp = '1e-08'

angle30 = 30.0 * np.pi/180.0

fieldpath = f'../../data/nonlinear_data/internal_damping/k_stretch_20.59_k_shear_0.83_sweeping_k_rot'
forcepath = f'../../data/nonlinear_data/reaction_forces/sim_data_harmonic_input_sweeping_krot/30_deg_sample'

def load_nonlin_varykrot_funcn(freq, angle):
    pow_ = 0
    tpts = np.load(f'{fieldpath}/{angle*180/np.pi:.0f}_deg_{freq}.0Hz_{amp}mm_10periods_sine_krotx2power{pow_}/timepoints.npz')['arr_0']

    fields = np.zeros((len(i_range), len(tpts), 2, geometry.n_blocks, 3))  # Initialize array to hold fields for all k_rot
    forces = np.zeros((len(i_range), len(tpts), geometry.n_blocks, 3))     # Initialize array to hold forces for all k_rot
    
    for idx, pow in enumerate(i_range):
        fields[idx] = np.load(f'{fieldpath}/{angle*180/np.pi:.0f}_deg_{freq}.0Hz_{amp}mm_10periods_sine_krotx2power{pow}/{angle30*180/np.pi:.0f}_deg_fields.npz')['arr_0']
        forces[idx] = np.load(f'{forcepath}/{freq}Hz/krotx2power{pow}_amplitude_{amp}_10periods_sine_reaction_forces.npz')['arr_0']

    return tpts, fields, forces


In [ ]:
# Setup analysis tools for varying parameter

k_rot_arr = exp_config['k_rot'] *  2.0**i_range

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle30,
    k_stretch=exp_config['k_stretch'],
    k_rot=k_rot_arr,
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/nonlinear_varying_krot')

d = save_path / 'map_norms'
d.mkdir(parents=True, exist_ok=True)

for idx, pow in enumerate(i_range):
    np.save(f'{save_path}/map_norms/krotx2power{pow}.npy',  tools['map_norm'][idx])

for i, freq in enumerate(lowamp_freqs):
    print(f"Processing frequency: {freq} Hz ({i+1}/{len(lowamp_freqs)})")
    
    tpts, fields, forces = load_nonlin_varykrot_funcn(freq)
    
    # Process each k_rot dataset
    for idx, pow in enumerate(i_range):
        print(f"  Analyzing k_rot x 2^{pow} ({idx+1}/{len(i_range)})")
        
        result = analyse_dataset_conservation(
            tpts,
            fields=fields[idx],
            forces=forces[idx],
            tools=slice_tools_for_param(tools, idx),
        )
        # print(f"map_ext_forces shape: {result['map_ext_forces'].shape}")
        # print(f"map_ext_forces dtype: {result['map_ext_forces'].dtype}")
        # print(f"map_ext_forces sample values: {result['map_ext_forces'][:5, :5]}")
        # print(f"Any NaN? {np.isnan(result['map_ext_forces']).any()}")
        
        # Save results (ensure directories exist once per save target)
        mom_dir = save_path / 'map_momenta' / f'krotx2power{pow}_amplitude_{amp}mm'
        momrate_dir = save_path / 'map_momentum_rates' / f'krotx2power{pow}_amplitude_{amp}mm'
        forces_dir = save_path / 'map_ext_forces' / f'krotx2power{pow}_amplitude_{amp}mm'
        for d in (mom_dir, momrate_dir, forces_dir):
            d.mkdir(parents=True, exist_ok=True)
        
        np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
        np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
        np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
        
    print(f"Finished processing frequency: {freq} Hz")

# Experiment

In [ ]:
exp_freqs = np.arange(2, 36, 1)

angle30 = 30.0 * np.pi/180.0

fieldpath = f'../../data/nonlinear_data/exp'
forcepath = f'../../data/nonlinear_data/reaction_forces/raw_exp_data'


def load_exp_funcn(freq, angle):
    tpts = np.load(f'{fieldpath}/{freq}Hz/timepoints.npz')['arr_0']
    fields = np.load(f'{fieldpath}/{freq}Hz/{angle*180/np.pi:.0f}_deg_fields.npz')['arr_0']
    forces = np.load(f'{forcepath}/{angle*180/np.pi:.0f}_deg_sample/{freq}Hz/reaction_forces.npz')['arr_0']

    return tpts, fields, forces


tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle30,
    k_stretch=exp_config['k_stretch'],
    k_rot=exp_config['k_rot'],
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])
print(tools['map_norm'])

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/raw_experiment_30_deg_sample')

for i, freq in enumerate(exp_freqs):
    
    tpts, fields, forces = load_exp_funcn(freq, angle30)

    result = analyse_dataset_conservation(
        tpts,
        fields=fields,
        forces=forces,
        tools=tools,
    )
    
    # Save results (ensure directories exist once per save target)
    mom_dir = save_path / 'map_momenta'
    momrate_dir = save_path / 'map_momentum_rates'
    forces_dir = save_path / 'map_ext_forces'
    for d in (mom_dir, momrate_dir, forces_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
    np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
    np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
        
    print(f"Finished processing frequency: {freq} Hz")


np.save(f'{save_path}/map_norms.npy',  tools['map_norm'])

In [ ]:
exp_freqs = np.arange(2, 36, 1)

angle5 = 5.0 * np.pi/180.0

fieldpath = f'../../data/nonlinear_data/exp'
forcepath = f'../../data/nonlinear_data/reaction_forces/raw_exp_data'


def load_exp_funcn(freq, angle):
    tpts = np.load(f'{fieldpath}/{freq}Hz/timepoints.npz')['arr_0']
    fields = np.load(f'{fieldpath}/{freq}Hz/{angle*180/np.pi:.0f}_deg_fields.npz')['arr_0']
    forces = np.load(f'{forcepath}/{angle*180/np.pi:.0f}_deg_sample/{freq}Hz/reaction_forces.npz')['arr_0']

    return tpts, fields, forces


tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle5,
    k_stretch=exp_config['k_stretch'],
    k_rot=exp_config['k_rot'],
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/raw_experiment_5_deg_sample')

for i, freq in enumerate(exp_freqs):
    
    tpts, fields, forces = load_exp_funcn(freq, angle5)

    result = analyse_dataset_conservation(
        tpts,
        fields=fields,
        forces=forces,
        tools=tools,
    )
    
    # Save results (ensure directories exist once per save target)
    mom_dir = save_path / 'map_momenta'
    momrate_dir = save_path / 'map_momentum_rates'
    forces_dir = save_path / 'map_ext_forces'
    for d in (mom_dir, momrate_dir, forces_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
    np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
    np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
        
    print(f"Finished processing frequency: {freq} Hz")


np.save(f'{save_path}/map_norms.npy',  tools['map_norm'])

# Linear Simulation (data needs to be generated using dilation_metamaterial_LinearRS_relaxing_regime.ipynb)

In [ ]:
angle30 = 30.0 * np.pi/180.0

linrelax_exp_path = rf'../../data/linear_data/linear_response_relaxing_regime/24x16_{angle30*180/np.pi:.1f}_expdriveend'

timepoints = np.load(f'{linrelax_exp_path}/timepoints.npy')
all_response = np.load(f'{linrelax_exp_path}/response.npy')
all_response_rate = np.load(f'{linrelax_exp_path}/response_rate.npy')

print(all_response.shape)

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle30,
    k_stretch=exp_config['k_stretch'],
    k_rot=exp_config['k_rot'],
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])
print(tools['map_norm'])

FileNotFoundError: [Errno 2] No such file or directory: '../../data/linear_data/linear_response_relaxing_regime/24x16_30.0_expdriveend/timepoints.npy'

In [ ]:
n1 = exp_config['n1_blocks']
n2 = exp_config['n2_blocks']
N = exp_config['N']

#Geometry
geometry = RotatedSquareGeometry(n1_cells=n1//2, n2_cells=n2//2, spacing=exp_config['spacing'], bond_length=exp_config['hinge_length'])
# centroid_node_vectors is a function of the initial angle
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()

# Setup parameters
greenshim_params = MechanicalParams(
        bond_params=LigamentParams(
            reference_vector=reference_bond_vectors(),
            k_stretch=exp_config['k_stretch'],
            k_shear=exp_config['k_shear'],
            k_rot=exp_config['k_rot'],
        ),
        density=exp_config['density'],
    )
control_params = ControlParams(
    geometrical_params=GeometricalParams(
        block_centroids=block_centroids(angle30),
        centroid_node_vectors=centroid_node_vectors(angle30),
        ),
    mechanical_params=greenshim_params,
    )  

damp_coeffs = np.array([exp_config['n_stretch'], exp_config['n_shear'], exp_config['n_rot']])

# Force coefficients
_stiffness, _damping = get_stiffness_damping_matrices(
    geometry=geometry,
    control_params=control_params,
    damp_coeffs=damp_coeffs
)

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/linear_30_deg_sample')

for i, freq in enumerate(exp_freqs):

    fields = all_response[i].reshape(all_response[i].shape[0], 2, -1)
    fields_rate = all_response_rate[i].reshape(all_response_rate[i].shape[0], 2, -1)

    result = analyse_dataset_conservation(
        timepoints,
        fields=fields,
        stiffness=_stiffness,
        damping=_damping,
        tools=tools,
        fields_rate=fields_rate
    )
    
    # Save results (ensure directories exist once per save target)
    mom_dir = save_path / 'map_momenta'
    momrate_dir = save_path / 'map_momentum_rates'
    forces_dir = save_path / 'map_ext_forces'
    for d in (mom_dir, momrate_dir, forces_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
    np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
    np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
    
    print(f"Finished processing frequency: {freq} Hz")

np.save(f'{save_path}/map_norms.npy',  tools['map_norm'])

In [ ]:
angle5 = 5.0 * np.pi/180.0

linrelax_exp_path = rf'../../data/linear_data/linear_response_relaxing_regime/24x16_{angle5*180/np.pi:.1f}_expdriveend'

timepoints = np.load(f'{linrelax_exp_path}/timepoints.npy')
all_response = np.load(f'{linrelax_exp_path}/response.npy')
all_response_rate = np.load(f'{linrelax_exp_path}/response_rate.npy')

print(all_response.shape)

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle5,
    k_stretch=exp_config['k_stretch'],
    k_rot=exp_config['k_rot'],
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])
print(tools['map_norm'])

(34, 2000, 2304)


NameError: name 'geometry' is not defined

In [ ]:
n1 = exp_config['n1_blocks']
n2 = exp_config['n2_blocks']
N = exp_config['N']

#Geometry
geometry = RotatedSquareGeometry(n1_cells=n1//2, n2_cells=n2//2, spacing=exp_config['spacing'], bond_length=exp_config['hinge_length'])
# centroid_node_vectors is a function of the initial angle
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()

# Setup parameters
greenshim_params = MechanicalParams(
        bond_params=LigamentParams(
            reference_vector=reference_bond_vectors(),
            k_stretch=exp_config['k_stretch'],
            k_shear=exp_config['k_shear'],
            k_rot=exp_config['k_rot'],
        ),
        density=exp_config['density'],
    )
control_params = ControlParams(
    geometrical_params=GeometricalParams(
        block_centroids=block_centroids(angle5),
        centroid_node_vectors=centroid_node_vectors(angle5),
        ),
    mechanical_params=greenshim_params,
    )  

damp_coeffs = np.array([exp_config['n_stretch'], exp_config['n_shear'], exp_config['n_rot']])

# Force coefficients
_stiffness, _damping = get_stiffness_damping_matrices(
    geometry=geometry,
    control_params=control_params,
    damp_coeffs=damp_coeffs
)

In [ ]:
save_path = Path(f'../../data/conformalanalysis_data/conservation_analysis/linear_5_deg_sample')

for i, freq in enumerate(exp_freqs):

    fields = all_response[i].reshape(all_response[i].shape[0], 2, -1)
    fields_rate = all_response_rate[i].reshape(all_response_rate[i].shape[0], 2, -1)

    result = analyse_dataset_conservation(
        timepoints,
        fields=fields,
        stiffness=_stiffness,
        damping=_damping,
        tools=tools,
        fields_rate=fields_rate
    )
    
    # Save results (ensure directories exist once per save target)
    mom_dir = save_path / 'map_momenta'
    momrate_dir = save_path / 'map_momentum_rates'
    forces_dir = save_path / 'map_ext_forces'
    for d in (mom_dir, momrate_dir, forces_dir):
        d.mkdir(parents=True, exist_ok=True)
    
    np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'], allow_pickle=False)
    np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'], allow_pickle=False)
    np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'], allow_pickle=False)
    
    print(f"Finished processing frequency: {freq} Hz")


np.save(f'{save_path}/map_norms.npy',  tools['map_norm'])

# Linear Simulation Varying $k_{\theta}$

In [ ]:
exp_freqs = np.arange(2, 36, 1)
i_range = np.arange(-8, 6)
angle30 = 30.0 * np.pi/180.0

fieldpath = '../../data/linear_data/linear_response_relaxing_regime/varyingkrotnrot_24x16_30.0_expdriveend'

tpts = np.load(f'{fieldpath}/timepoints.npy')

def load_response_funcn(p):
    resp = np.load(f'{fieldpath}/krotnrot_x_pow(2,{p})_response.npy')
    resp_rate = np.load(f'{fieldpath}/krotnrot_x_pow(2,{p})_responserate.npy')
    return resp, resp_rate

In [ ]:
# Setup analysis tools for varying parameter

k_rot_arr = exp_config['k_rot'] *  2.0**i_range
n_rot_arr = exp_config['n_rot'] *  2.0**i_range

tools = tools_for_conservation_analysis(
    geometry=geometry,
    initial_angle=angle30,
    k_stretch=exp_config['k_stretch'],
    k_rot=k_rot_arr,
    spacing=exp_config['spacing'],
    hinge_length=exp_config['hinge_length'],
    density=exp_config['density']
)

print("Map expressions for conservation analysis : ", tools['mono_exprs'])

In [ ]:
save_path = Path('../../data/conformalanalysis_data/conservation_analysis/linear_varyingkrot')

for j, p in enumerate(i_range):
    print(f"Analyzing k_rot x 2^{p} ({j+1}/{len(i_range)})")
    
    fields, fields_rate = load_response_funcn(p)
    fields = fields.reshape(fields.shape[0], fields.shape[1], 2, -1)
    fields_rate = fields_rate.reshape(fields_rate.shape[0], fields_rate.shape[1], 2, -1)

    _stiffness, _damping = get_stiffness_damping_matrices(
        geometry=geometry,
        control_params=ControlParams(
            geometrical_params=GeometricalParams(
                block_centroids=block_centroids(angle30),
                centroid_node_vectors=centroid_node_vectors(angle30),
                ),
            mechanical_params=MechanicalParams(
                bond_params=LigamentParams(
                    reference_vector=reference_bond_vectors(),
                    k_stretch=exp_config['k_stretch'],
                    k_shear=exp_config['k_shear'],
                    k_rot=k_rot_arr[j],
                ),
                density=exp_config['density'],
            )
        ),
        damp_coeffs= np.array([exp_config['n_stretch'], exp_config['n_shear'], n_rot_arr[j]])
    )
    
    # Process each frequency dataset
    for i, freq in enumerate(exp_freqs):

        result = analyse_dataset_conservation(
            tpts,
            fields=fields[i],
            tools=slice_tools_for_param(tools, j),
            fields_rate=fields_rate[i],
            stiffness=_stiffness,
            damping=_damping  
        )
        
        # Save results (ensure directories exist once per save target)
        mom_dir = save_path / 'map_momenta' / f'krotx2power{p}'
        momrate_dir = save_path / 'map_momentum_rates' / f'krotx2power{p}'
        forces_dir = save_path / 'map_ext_forces' / f'krotx2power{p}'
        for d in (mom_dir, momrate_dir, forces_dir):
            d.mkdir(parents=True, exist_ok=True)
        
        np.save(mom_dir / f'{freq}Hz.npy', result['map_momentum'])
        np.save(momrate_dir / f'{freq}Hz.npy', result['map_momentum_rate'])
        np.save(forces_dir / f'{freq}Hz.npy', result['map_ext_forces'])
        
    print(f"Finished processing")


d = save_path / 'map_norms'
d.mkdir(parents=True, exist_ok=True)

for idx, pow in enumerate(i_range):
    np.save(f'{save_path}/map_norms/krotx2power{pow}.npy',  tools['map_norm'][idx])